In [0]:
%run ../00-common/01.environment-config


In [0]:
bronze_table = f"{catlog_name}.{bronze_schema}.drivers"
silver_table = f"{catlog_name}.{silver_schema}.drivers"

In [0]:
driver_df = spark.read.table(bronze_table)

In [0]:
display(driver_df)

In [0]:
driver_selected_df = driver_df.drop("url")

In [0]:
display(driver_selected_df)

In [0]:
driver_renamed_df = (driver_selected_df
                     .withColumnsRenamed({
                         "driverId": "driver_id",
                         "dateOfBirth": "driver_date_of_birth"
                     }))

In [0]:
from pyspark.sql import functions as F

In [0]:
driver_renamed_df = (
    driver_renamed_df
    .withColumn('driver_name',
                F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName")))

)
    .drop("name"))

In [0]:
display(driver_renamed_df)

In [0]:
driver_distinct_df = driver_renamed_df.dropDuplicates(["driver_id"])


In [0]:
display(driver_distinct_df)

In [0]:
driver_final_df = (driver_distinct_df
                   .withColumn("nationality", F.initcap(F.col("nationality"))))

In [0]:
display(driver_final_df)

In [0]:
(
    driver_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))